## 在 W&B 中可视化嵌入

我们将把数据上传到 [Weights & Biases](http://wandb.ai)，并使用 [Embedding Projector（嵌入投影器）](https://docs.wandb.ai/ref/app/features/panels/weave/embedding-projector)，借助 PCA、UMAP 和 t-SNE 等常见降维算法来可视化嵌入。数据集由 [Get_embeddings_from_dataset Notebook](../Get_embeddings_from_dataset.ipynb) 生成。

## 什么是 Weights & Biases？

[Weights & Biases](http://wandb.ai) 是一个机器学习平台，OpenAI 及其他机器学习团队使用它来更快地构建更好的模型。他们用它快速跟踪实验、评估模型性能、复现模型、可视化结果，并与同事分享发现。

### 1. 将数据记录到 W&B

我们创建一个包含原始数据和嵌入的 [W&B Table](https://docs.wandb.ai/guides/data-vis/log-tables)。每条评论是一行新数据，1536 个嵌入浮点数各自拥有一列，列名为 `emb_{i}`。

In [2]:
import pandas as pd
from sklearn.manifold import TSNE
import numpy as np
from ast import literal_eval

# Load the embeddings
datafile_path = "data/fine_food_reviews_with_embeddings_1k.csv"
df = pd.read_csv(datafile_path)

# Convert to a list of lists of floats
matrix = np.array(df.embedding.apply(literal_eval).to_list())

In [ ]:
import wandb

original_cols = df.columns[1:-1].tolist()
embedding_cols = ['emb_'+str(idx) for idx in range(len(matrix[0]))]
table_cols = original_cols + embedding_cols

with wandb.init(project='openai_embeddings'):
    table = wandb.Table(columns=table_cols)
    for i, row in enumerate(df.to_dict(orient="records")):
        original_data = [row[col_name] for col_name in original_cols]
        embedding_data = matrix[i].tolist()
        table.add_data(*(original_data + embedding_data))
    wandb.log({'openai_embedding_table': table})

### 2. 渲染为二维投影

打开 W&B 运行链接后，点击 Table 右上角的 ⚙️ 图标，并将 "Render As:" 改为 "Combined 2D Projection"。 

示例：http://wandb.me/openai_embeddings